# Solar Filament Segmentation Challenge 2026

## Genuine, instance-aware MAGFiLO pipeline

This notebook trains and evaluates a model from the supplied training images and COCO annotations only. Test predictions are generated by tiled model inference and encoded at runtime; there is no precomputed RLE payload, hidden-label lookup, fallback mask, or leaderboard-derived artifact.


## 1. Forensic audit of the legacy 0.55 notebook

The supplied legacy notebook's reported score cannot be attributed to its small U-Net: its captured validation Dice is approximately zero while its submission is emitted from `CHAMPION_PAYLOAD` precomputed RLEs rather than model logits. Its other material weaknesses are a 60-image subset, five CPU epochs, random 256-pixel crops, simplistic normalization, BCE+Dice-only optimization, no observation-grouped validation, no instance targets/PQ evaluator, no threshold sweep, no TTA, and no morphology-aware reconstruction. The payload and fallback-mask route are deliberately not reproduced here.

**Diagnostic rule:** before interpreting a low validation score, inspect 20 aligned COCO masks, foreground prevalence, split instance counts, and threshold response. This distinguishes rasterization/split failures from training collapse.


## 2. Design and reproducibility

Version 1 is a ResNet-34 U-Net-style semantic/boundary model trained on 768-pixel filament-aware crops with a weighted BCE + Dice + Tversky objective. Version 2 validates connected-component and watershed reconstruction by PQ. TTA and ensemble hooks are intentionally optional and must be retained only after a measured validation improvement. Model selection is PQ first, then Dice/IoU and merge/split diagnostics.

Run the notebook on Kaggle after attaching the competition dataset. It dynamically discovers its paths; it does not assume an exact dataset slug.


In [ ]:
from dataclasses import asdict
from pathlib import Path
import json
import logging
import torch

from solar_filament.pipeline import (
    CONFIG, PipelineConfig, SolarFilamentDataset, FilamentSegmentationModel,
    append_experiment, create_submission, environment_report, find_competition_data,
    group_train_validation_split, load_coco_annotations, run_synthetic_metric_tests,
    set_seed, validate,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
config = PipelineConfig(debug_mode=False, max_train_images=None)
set_seed(config.seed)
print(json.dumps(environment_report(config), indent=2))
run_synthetic_metric_tests()


## 3. Discover, parse, and audit annotations

`build_instance_masks`, `build_semantic_mask`, and `build_instance_id_mask` in the source module decode both COCO polygon and RLE annotations and preserve every instance. Before training, render 20 randomly selected images/masks with the helper code appropriate to your environment and confirm coordinate alignment. Do not continue if masks are empty or offset.


In [ ]:
data = find_competition_data('/kaggle/input')
index = load_coco_annotations(data.annotation_json)
train_ids, val_ids = group_train_validation_split(index, config.val_fraction, config.seed)
if config.debug_mode:
    train_ids, val_ids = train_ids[:8], val_ids[:4]
assert set(train_ids).isdisjoint(val_ids)

# Full data is the default. This value should only change in explicit fast-development runs.
print('Using', len(train_ids), 'training observations and', len(val_ids), 'validation observations.')


## 4. Training (Version 1 baseline)

Use `train_one_epoch` with a `DataLoader`, AdamW, cosine schedule, AMP, and save checkpoints by measured validation PQ/Dice. This cell is intentionally a template: it has no fabricated metric output. Record every completed run in `experiments.csv` with `append_experiment`.


In [ ]:
# Example training setup. Execute only when GPU runtime and data are available.
# from torch.utils.data import DataLoader
# from solar_filament.pipeline import train_one_epoch
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# train_ds = SolarFilamentDataset(index, data.train_images, train_ids, config, training=True)
# val_ds = SolarFilamentDataset(index, data.train_images, val_ids, config, training=False)
# model = FilamentSegmentationModel(config).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
# scaler = torch.amp.GradScaler('cuda', enabled=config.use_amp and device.type == 'cuda')
# loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers, pin_memory=device.type == 'cuda')
# best_pq = -1.0
# for epoch in range(config.epochs):
#     loss = train_one_epoch(model, loader, optimizer, scaler, config, device, pos_weight=5.0)  # measure prevalence first; replace 5.0
#     metrics = validate(model, val_ds, config, device)
#     scheduler.step()
#     print(epoch + 1, loss, metrics)
#     if metrics['global_pq'] > best_pq:
#         best_pq = metrics['global_pq']; torch.save({'model': model.state_dict(), 'config': asdict(config), 'metrics': metrics}, '/kaggle/working/best_model.pth')


## 5. PQ-led post-processing, final inference, and submission

Run threshold/min-area/watershed selection only on validation probability maps. Then load the selected checkpoint. `create_submission` performs overlapping full-resolution inference, optional invertible TTA, conservative solar-disk suppression, instance reconstruction, Fortran-order COCO RLE encoding, and per-mask round-trip verification. It does not create an arbitrary mask for images with no detections.


In [ ]:
# checkpoint = torch.load('/kaggle/working/best_model.pth', map_location=device)
# model.load_state_dict(checkpoint['model'])
# with open('/kaggle/working/config.json', 'w') as f: json.dump(asdict(config), f, indent=2)
# with open('/kaggle/working/validation_results.json', 'w') as f: json.dump(checkpoint['metrics'], f, indent=2)
# submission = create_submission(model, data, config, '/kaggle/working/submission.csv', device)
# append_experiment('/kaggle/working/experiments.csv', 'resnet34_boundary_v1', config, checkpoint['metrics'], runtime_seconds=0.0)
# assert list(submission.columns) == ['filament_id', 'segmentation_rle']


## 6. Reporting policy

Do not claim any metric until the validation cell computes it from the held-out observation split. Preserve failures in `experiments.csv`; retain TTA, watershed, a boundary head, or an ensemble only if the untouched validation analysis supports it. The generated report should state architecture, resolution, epochs, losses, preprocessing, augmentations, tiling/TTA, reconstruction settings, measured Dice/IoU/PQ, instance diagnostics, and runtime.
